# ⏳ Kaggriculture Ladder: How Many Days Until Your Score is Real?

**Author**: Vijai K Mathew  
**Competition**: [Kaggriculture](https://www.kaggle.com/competitions/kaggriculture)  
**Analysis Date**: 2026-08-24  
**Data**: 15 submissions (v30–v44), 231 episodes via Kaggle MCP `list_submission_episodes`, 6,185-team public leaderboard  

---

## 📌 TL;DR — How Many Days?

> **A submission needs ~3 days (72 hours) and ~100 ranked games to reach its stable rating.**  
> The first **6 hours do 70% of the climb** (~72 games).  
> After **24 hours (~98 games)** huge jumps stop (>100 points unlikely).  
> After **48–72 hours** movement is noise (<50/day) unless the meta shifts.  
> **Judge at 24h / 60 games; trust at 72h / 100 games. Never re-roll before your tier hold expires.**

| Window | Cumulative Games (avg) | Score Jump | Huge Jump? |
|---|---|---|---|
| **0–1 hr** | ~15 | **+850** (600→~1500) | YES |
| **1–6 hr** | **~72** | **+450** (~1500→~2000) | **YES — 70% of climb** |
| **6–12 hr** | ~81 | +120 | moderate |
| **12–24 hr** | ~98 | +80 | small |
| **24–72 hr** | ~101 | <50 total | **NO — stable** |

_No submission code is shown in this notebook — only public ladder data and episode timing._


In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# This notebook uses ONLY public leaderboard + episode timing via Kaggle MCP.
# It contains ZERO submission source code — analysis only.
# Synthetic fallback matches the mined trajectory report (report-2026-08-23.md)
np.random.seed(42)
window_stats = pd.DataFrame({
    'window': ['0-1h','1-6h','6-12h','12-24h','24-36h','36-48h','48-72h'],
    'cum_games': [15, 72, 81, 98, 101, 101, 101],
    'cum_wr': [0.93, 0.71, 0.67, 0.65, 0.63, 0.63, 0.63],
    'score_delta': [850, 450, 120, 80, 30, 15, 10],
    'games_in_window': [15, 57, 9, 17, 3, 0, 0]
})
window_stats


## 1. 🏗️ Ladder Mechanics — Why 3 Days?

- Only your **latest 2 submissions** are matched on the ladder. Every new submit **evicts the oldest active one** — its accumulated rating is abandoned.
- A fresh submission restarts at baseline (~600) and must re-run the calibration burst. Each submit costs **one slot × ~72h of climb time**.
- Matchmaking feeds weak opponents first (validation + low-rated teams), then promotes you to mid-tier, then top-200. Win rate falls 1.00 → 0.60 as rating climbs — **that fall IS the climb**.
- Cadence decays: ~15/hr (first hour) → ~11/hr (next 5h) → 1.8/hr (6–12h) → 0.2/hr (24–72h). The scheduler throttles you as rating rises.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Games vs time
axes[0].bar(window_stats['window'], window_stats['cum_games'], color='#0284c7', alpha=0.85)
axes[0].set_title('Cumulative Ranked Games vs Time', fontweight='bold')
axes[0].set_ylabel('Cumulative Games')
axes[0].set_xlabel('Elapsed Time Window')
for i, v in enumerate(window_stats['cum_games']):
    axes[0].text(i, v+2, str(v), ha='center', fontsize=9)

# Score delta per window
axes[1].bar(window_stats['window'], window_stats['score_delta'], color='#e11d48', alpha=0.85)
axes[1].set_title('Score Jump per Window (Avg, Elo-like)', fontweight='bold')
axes[1].set_ylabel('Score Delta in Window')
axes[1].set_xlabel('Elapsed Time Window')
for i, v in enumerate(window_stats['score_delta']):
    axes[1].text(i, v+15, f'+{v}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()
print('Knee at 6h: 70% of total climb done. Flat after 24h.')


## 2. 📉 Win-Rate Decay IS the Rating Climb

Mean cumulative win rate across mined submissions: `1.00 (15 games) → 0.76 (50) → 0.69 (80) → 0.63 (100) → 0.60 (stable)`.  
If your WR stays >0.60 at 100 games, you are underrated and will climb further.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
xs = [15, 50, 70, 80, 98, 101]
ys = [1.00, 0.76, 0.71, 0.69, 0.65, 0.63]
ax.plot(xs, ys, marker='o', color='#0ea5e9', linewidth=2.5)
ax.fill_between(xs, ys, alpha=0.15, color='#0ea5e9')
ax.set_title('Win-Rate Decay vs Games Played (Mean)', fontweight='bold')
ax.set_xlabel('Cumulative Ranked Games')
ax.set_ylabel('Cumulative Win Rate')
ax.set_ylim(0.5, 1.02)
ax.grid(True, alpha=0.3)
for x,y in zip(xs, ys):
    ax.text(x, y+0.02, f'{y:.2f}', ha='center', fontsize=8)
plt.show()


## 3. 🛡️ Tiered Holding Policy — More Rank, More Hours

Convergence says ~72h stabilizes — but that investment is only worth protecting when rating is high. **More rank, more hours** (capped at top-50 = 7 days):

| Tier | Public Score | Min Hold | Verdict at Expiry |
|---|---|---|---|
| Probe | <1800 | 24h | PROBE-EVICT |
| Mover | 1800–2200 | 48h | RE-ROLL if stalled |
| Contender | 2200–2396 | 72h | RE-ROLL if stalled |
| Top 200 | ≥2396 | 96h (4d) | RE-ROLL only on stall |
| Top 100 | ≥2518 | 120h (5d) | RE-ROLL only on stall |
| Top 50+ | ≥2628 | 168h (7d) | RE-ROLL only on stall |

_Anchors from live leaderboard on 2026-08-23; refresh as ladder inflates._


In [ ]:
hold = pd.DataFrame({
    'tier': ['Probe','Mover','Contender','Top 200','Top 100','Top 50+'],
    'hold_h': [24, 48, 72, 96, 120, 168],
    'cutoff': [0, 1800, 2200, 2396, 2518, 2628]
})
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(hold['tier'], hold['hold_h'], color=['#94a3b8','#38bdf8','#0ea5e9','#0284c7','#0369a1','#0c4a6e'])
ax.set_title('Minimum Hold Duration by Tier (Hours)', fontweight='bold')
ax.set_xlabel('Hours')
for i, v in enumerate(hold['hold_h']):
    ax.text(v+2, i, f'{v}h', va='center', fontsize=9)
plt.tight_layout()
plt.show()


## 4. 🔍 How Many Replays to Analyze?

| Window | Episodes Available | Replays to Analyze | Why |
|---|---|---|---|
| 1h | 15 | **5** | High variance, weak opponents — learn little |
| 6h | 70 | **12** | Best ROI — weak→mid transition |
| 24h | 98 | **20** | **Gold standard** — <5% WR error |
| 72h | 101 | **5** (losses vs top-50 only) | Noise beyond this |


## 5. 🧰 Using Kaggle MCP for This Analysis (No Code Leakage)

| Task | MCP Tool (preferred) | When |
|---|---|---|
| List submissions | `kaggle_search_competition_submissions` | Every report |
| List episodes per submission | `kaggle_list_submission_episodes` | Per-window stats |
| Get episode replay | `kaggle_get_episode_replay` | Per-turn EDA |
| Leaderboard | `kaggle_get_competition_leaderboard` | Cutoffs, rank |

**MCP gaps:** episode list returns ~30 latest via API pagination; for full history loop pageToken or use `tools/scheduler.py run download` (caches `data/downloads/<id>/episodes.csv`).

---

### Reproduce Locally
```bash
uv run python tools/scheduler.py run download  # caches episodes.csv
# window stats via parse_time(create_time) - parse_time(submission_date)
```

### Competition Rules
- **5 submissions/day max**, **2 tracked slots (latest 2)**. Max team 5. No private leaderboard.
- **Deadlines:** new entrant 2026-09-23, final 2026-09-30 23:59 UTC (~37 days remain).
- **Final judging:** top 2 latest submissions are final — ensure best 2 are newest before 09-30.


## ✅ Takeaway

- Treat a submission as a **72-hour investment**, not an instant score.
- After 24h/60 games huge jumps stop; after 72h/100 games it's stable.
- Submitting to get *more* replays is a myth — it evicts and resets. Submit only champion-grade agents.
- Never batch >2; never judge before 24h; never trust before 72h.

_Data: report-2026-08-23.md (15 subs, 231 episodes) + ladder-convergence-2026-08-23.md. No agent source shown._
